# Cluster 2 Stacking Model - Alternative Implementation

## Objective
Build an independent stacking model for Cluster 2 using a different architectural approach to provide model diversity.

In [68]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import recall_score, make_scorer, confusion_matrix, classification_report
from sklearn.metrics import average_precision_score, f1_score, precision_score

from sklearn.ensemble import RandomForestClassifier, StackingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from lightgbm import LGBMClassifier

RANDOM_STATE = 42

In [69]:

df_cluster2 = pd.read_csv("cluster_2.csv")
print(f"Data loaded. Shape: {df_cluster2.shape}")


top40 = joblib.load("top_features_for_clustering.joblib")
features_to_use = top40

X_sub = df_cluster2[features_to_use].copy()
y_sub = df_cluster2["Bankrupt?"].copy()

print(f"\nFeatures: {X_sub.shape}")
print(f"Target Distribution:\n{y_sub.value_counts()}")
print(f"\nPositive class ratio: {y_sub.sum() / len(y_sub):.4%}")
print(f"Class imbalance ratio: {(y_sub == 0).sum() / (y_sub == 1).sum():.1f}:1")

Data loaded. Shape: (2059, 98)

Features: (2059, 40)
Target Distribution:
Bankrupt?
0    2053
1       6
Name: count, dtype: int64

Positive class ratio: 0.2914%
Class imbalance ratio: 342.2:1


## Base Model Configuration

Using diverse base models with appropriate imbalance handling

In [70]:

scale_pos_weight = (y_sub == 0).sum() / (y_sub == 1).sum()
print(f"Scale pos weight for imbalance: {scale_pos_weight:.2f}")

base_estimators = []


rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
base_estimators.append(('rf', rf))


et = ExtraTreesClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
base_estimators.append(('et', et))


svc = SVC(
    probability=True,
    class_weight='balanced',
    random_state=RANDOM_STATE
)
base_estimators.append(('svc', svc))

print(f"\nTotal base models: {len(base_estimators)}")

Scale pos weight for imbalance: 342.17

Total base models: 3


## Meta Model Configuration

Simple GradientBoosting classifier to avoid overfitting

In [71]:
meta_model = LogisticRegression(
    class_weight='balanced',
    C=0.5,
    random_state=RANDOM_STATE
)

## Build Stacking Pipeline

Using passthrough=True to provide meta-model access to original features

In [72]:

from sklearn.model_selection import StratifiedKFold

cv_stacking = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)


stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_model,
    cv=cv_stacking,
    stack_method='predict_proba',
    passthrough=False,
    n_jobs=-1
)


model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('stacking', stacking_clf)
])

## Train Final Model

In [73]:
model_pipeline.fit(X_sub, y_sub)

y_pred = model_pipeline.predict(X_sub)
y_pred_proba = model_pipeline.predict_proba(X_sub)[:, 1]

cm = confusion_matrix(y_sub, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

TT = int(tp)
TF = int(fn)
N_features = len(features_to_use)
eq1_acc = TT / (TF + TT) if (TF + TT) > 0 else 0.0


print(f"Confusion Matrix:")
print(f"  [[TN={tn:4d}, FP={fp:4d}]")
print(f"   [FN={fn:4d}, TP={tp:4d}]]")
print(f"\nTable 3 Metrics:")
print(f"  TT (True Bankrupts Caught): {TT}")
print(f"  TF (Bankrupts Missed):      {TF}")
print(f"  N_features:                 {N_features}")
print(f"  Eq(1) Accuracy (Recall):    {eq1_acc:.4f}")

Confusion Matrix:
  [[TN=2037, FP=  16]
   [FN=   0, TP=   6]]

Table 3 Metrics:
  TT (True Bankrupts Caught): 6
  TF (Bankrupts Missed):      0
  N_features:                 40
  Eq(1) Accuracy (Recall):    1.0000


## Additional Performance Metrics

In [74]:
precision = precision_score(y_sub, y_pred, zero_division=0)
f1 = f1_score(y_sub, y_pred, zero_division=0)
avg_precision = average_precision_score(y_sub, y_pred_proba)

print("\nAdditional Performance Metrics:")
print(f"  Precision:          {precision:.4f}")
print(f"  F1-Score:           {f1:.4f}")
print(f"  Average Precision:  {avg_precision:.4f}")

print("\nClassification Report:")
print(classification_report(y_sub, y_pred, target_names=['Non-Bankrupt', 'Bankrupt'], zero_division=0))


Additional Performance Metrics:
  Precision:          0.2727
  F1-Score:           0.4286
  Average Precision:  1.0000

Classification Report:
              precision    recall  f1-score   support

Non-Bankrupt       1.00      0.99      1.00      2053
    Bankrupt       0.27      1.00      0.43         6

    accuracy                           0.99      2059
   macro avg       0.64      1.00      0.71      2059
weighted avg       1.00      0.99      0.99      2059



## Save Model Package

In [75]:
cluster2_package = {
    "cluster_id": 2,
    "features": features_to_use,
    "pipeline": model_pipeline,
    "table3_stats": {
        "TT": TT, 
        "TF": TF, 
        "Eq1_acc": eq1_acc, 
        "N_features": N_features
    },
    "architecture": {
        "base_models": [name for name, _ in base_estimators],
        "meta_model": "LogisticRegression",
        "scaler": "StandardScaler",
        "stacking_cv": "StratifiedKFold(3)",
        "passthrough": False
    }
}

joblib.dump(cluster2_package, "cluster2_stacking_D.joblib")

['cluster2_stacking_D.joblib']